# Sentinel Cyber — Network Intrusion Detection

**Dataset:** CIC-IDS2017  
**Task:** Binary classification of network flows:
- `BENIGN` → `0`
- Any attack label → `1`

This notebook builds and evaluates a Random Forest baseline for the Sentinel Cyber demo.

## Experiment design

We use a **stratified random row split**:
- 70% training
- 15% validation
- 15% final test

The validation set is used to choose the decision threshold. The final test set is evaluated after that choice.

> **Interpretation warning:** A random row split can place highly similar flows, and potentially duplicate or related traffic, in multiple subsets. The strong scores below are results on this CIC-IDS2017 split; they do not establish performance on genuinely new networks, future traffic, or unseen attack families. This is a quick demo-oriented baseline, not production validation.

## 0. Setup

Run the notebook from top to bottom. KaggleHub may require Kaggle authentication in your environment.

In [1]:
%pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import gc
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 1. Download and locate CIC-IDS2017

The source filename is retained so the capture day remains available for EDA and later error analysis. It is not used as a model feature.

In [3]:
dataset_path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
csv_files = sorted(
    glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
)

print("Dataset directory:", dataset_path)
print("CSV files found:", len(csv_files))
for file in csv_files:
    print(os.path.basename(file))

if not csv_files:
    raise FileNotFoundError("No CSV files were found in the downloaded dataset.")

Dataset directory: C:\Users\Suhan Khan\.cache\kagglehub\datasets\chethuhn\network-intrusion-dataset\versions\1
CSV files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


## 2. Load and combine the CSV files

`Capture_Day` and `Source_File` are metadata columns. They help us understand where each flow came from, but are excluded from model inputs.

In [ ]:
def capture_day_from_filename(filename):
    name = os.path.basename(filename).lower()
    for day in ["monday", "tuesday", "wednesday", "thursday", "friday"]:
        if day in name:
            return day.capitalize()
    return "Unknown"

frames = []
for file in csv_files:
    part = pd.read_csv(file, low_memory=False)
    part.columns = part.columns.str.strip()
    part["Capture_Day"] = capture_day_from_filename(file)
    part["Source_File"] = os.path.basename(file)
    frames.append(part)
    print(f"{os.path.basename(file)}: {len(part):,} rows")

raw_df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()

print("\nCombined dataset shape:", raw_df.shape)
display(raw_df.head())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows


## 3. Inspect labels and capture-day distribution

In [ ]:
if "Label" not in raw_df.columns:
    raise KeyError("Expected a 'Label' column in CIC-IDS2017.")

raw_df["Label"] = raw_df["Label"].astype("string").str.strip()

print("Total rows:", f"{len(raw_df):,}")
print("Number of columns:", len(raw_df.columns))

print("\nRows by capture day:")
display(raw_df["Capture_Day"].value_counts().to_frame("rows"))

print("\nOriginal attack-label distribution:")
display(raw_df["Label"].value_counts(dropna=False).to_frame("rows"))

In [ ]:
plt.figure(figsize=(11, 5))
raw_df["Label"].value_counts().plot(kind="bar")
plt.title("CIC-IDS2017 original label distribution")
plt.xlabel("Original label")
plt.ylabel("Number of flows")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
plt.show()

## 4. Data quality checks and binary target

- Convert candidate feature columns to numeric values.
- Replace positive and negative infinity with missing values.
- Count exact duplicate rows, but do not automatically delete them.
- Create `Target`: benign is `0`; every non-benign label is `1`.
- Missing numeric values will be imputed inside the model pipeline.

In [ ]:
metadata_cols = ["Capture_Day", "Source_File"]
label_col = "Label"

duplicate_column_names = raw_df.columns[raw_df.columns.duplicated()].tolist()
print("Duplicate column names:", duplicate_column_names)
if duplicate_column_names:
    raise ValueError("Duplicate column names must be resolved before training.")

for col in raw_df.columns:
    if col not in metadata_cols + [label_col]:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

numeric_cols = raw_df.select_dtypes(include=np.number).columns
raw_df[numeric_cols] = raw_df[numeric_cols].replace([np.inf, -np.inf], np.nan)

print("Exact duplicate rows:", f"{raw_df.duplicated().sum():,}")
print("\nColumns with missing values (top 20):")
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0].head(20).to_frame("missing_values"))

raw_df["Target"] = (
    raw_df["Label"].str.upper().ne("BENIGN").fillna(True)
).astype("int8")

print("\nBinary target counts:")
display(
    raw_df["Target"]
    .value_counts()
    .rename(index={0: "BENIGN", 1: "ATTACK"})
    .to_frame("rows")
)
print("Attack percentage:", round(raw_df["Target"].mean() * 100, 3), "%")

## 5. Explore class balance by capture day

The original labels are concentrated unevenly across capture days. This table is descriptive; the model split below is a random stratified split rather than a day-based holdout.

In [ ]:
day_label_counts = pd.crosstab(raw_df["Capture_Day"], raw_df["Target"])
day_label_counts = day_label_counts.rename(columns={0: "BENIGN", 1: "ATTACK"})
display(day_label_counts)

ax = day_label_counts.plot(kind="bar", stacked=True, figsize=(9, 5))
ax.set_title("Benign and attack flows by capture day")
ax.set_xlabel("Capture day")
ax.set_ylabel("Number of flows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Feature engineering

Create directional packet and byte ratios, plus indicators for flows with no backward packets or bytes. A zero denominator becomes `NaN`; the training-fitted median imputer handles it later.

In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

if {"Total Fwd Packets", "Total Backward Packets"}.issubset(raw_df.columns):
    raw_df["Fwd_Bwd_Packet_Ratio"] = safe_divide(
        raw_df["Total Fwd Packets"],
        raw_df["Total Backward Packets"]
    )
    raw_df["No_Backward_Packets"] = (
        raw_df["Total Backward Packets"].eq(0).astype("int8")
    )

if {"Total Length of Fwd Packets", "Total Length of Bwd Packets"}.issubset(raw_df.columns):
    raw_df["Fwd_Bwd_Byte_Ratio"] = safe_divide(
        raw_df["Total Length of Fwd Packets"],
        raw_df["Total Length of Bwd Packets"]
    )
    raw_df["No_Backward_Bytes"] = (
        raw_df["Total Length of Bwd Packets"].eq(0).astype("int8")
    )

print("Feature engineering complete.")

## 7. Build numeric features and split the data

We exclude labels and metadata from the feature matrix. The split is stratified by the binary target so the benign/attack proportions remain similar across subsets.

**Split:** 70% training, 15% validation, 15% final test.

In [ ]:
excluded_columns = ["Label", "Target", "Capture_Day", "Source_File"]

feature_frame = raw_df.drop(
    columns=excluded_columns,
    errors="ignore"
).select_dtypes(include=np.number)

feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan)

# Remove columns that contain no usable values or are constant.
all_missing = feature_frame.columns[feature_frame.isna().all()].tolist()
constant = [
    col for col in feature_frame.columns
    if feature_frame[col].nunique(dropna=True) <= 1
]
drop_features = sorted(set(all_missing + constant))
feature_frame = feature_frame.drop(columns=drop_features)

feature_columns = feature_frame.columns.tolist()

print("Feature matrix shape:", feature_frame.shape)
print("Features removed as all-missing/constant:", drop_features)

In [ ]:
indices = np.arange(len(raw_df))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=raw_df["Target"]
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=raw_df["Target"].iloc[temp_idx]
)

X_fit = feature_frame.iloc[train_idx].copy()
X_val = feature_frame.iloc[val_idx].copy()
X_test = feature_frame.iloc[test_idx].copy()

y_fit = raw_df["Target"].iloc[train_idx].copy()
y_val = raw_df["Target"].iloc[val_idx].copy()
y_test = raw_df["Target"].iloc[test_idx].copy()

# Preserve original multiclass labels for attack-family analysis.
label_test = raw_df["Label"].iloc[test_idx].copy()

print("Training:", X_fit.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

for split_name, target in [
    ("Training", y_fit),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    print(f"\n{split_name} target distribution:")
    print(target.value_counts().rename(index={0: "BENIGN", 1: "ATTACK"}))

## 8. Train the Random Forest baseline

Median imputation is part of the pipeline, so the imputation values are learned from the training data only.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created.")

In [ ]:
start_time = time.time()
model.fit(X_fit, y_fit)
training_minutes = (time.time() - start_time) / 60

print(f"Training completed in {training_minutes:.2f} minutes.")

y_val_proba = model.predict_proba(X_val)[:, 1]
print("Validation predictions generated:", len(y_val_proba))

## 9. Select the decision threshold using validation data

A threshold of `0.05` means a flow is flagged as an attack when its predicted attack probability is at least 5%.

Selection rule: choose the **highest recall** among tested thresholds with validation precision of at least 80%. If no threshold meets that precision requirement, choose the threshold with the best F1 score.

The final test set is not used to choose the threshold.

In [ ]:
thresholds = np.round(np.arange(0.05, 1.00, 0.05), 2)
threshold_rows = []

for threshold in thresholds:
    val_pred = (y_val_proba >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))

eligible = threshold_df[threshold_df["Precision"] >= 0.80]

if not eligible.empty:
    best_row = eligible.sort_values(
        ["Recall", "F1", "Threshold"],
        ascending=[False, False, False]
    ).iloc[0]
    print("Selection: highest recall with validation precision >= 0.80")
else:
    best_row = threshold_df.sort_values(
        ["F1", "Precision", "Recall"],
        ascending=[False, False, False]
    ).iloc[0]
    print("No threshold met 80% precision; selected best F1.")

selected_threshold = float(best_row["Threshold"])
print(f"Selected threshold: {selected_threshold:.2f}")
print(f"Validation precision: {best_row['Precision']:.4f}")
print(f"Validation recall: {best_row['Recall']:.4f}")
print(f"Validation F1: {best_row['F1']:.4f}")

## 10. Final evaluation on the test set

This is the final evaluation for the chosen threshold. Do not keep adjusting the threshold based on these test results.

In [ ]:
y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= selected_threshold).astype(int)

test_metrics = {
    "Threshold": selected_threshold,
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Attack precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Attack recall": recall_score(y_test, y_test_pred, zero_division=0),
    "Attack F1": f1_score(y_test, y_test_pred, zero_division=0)
}

print("FINAL TEST RESULTS")
print("------------------")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, (float, np.floating)) else f"{metric}: {value}")

print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
print(cm)

print("\nClassification report:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=[0, 1],
    target_names=["BENIGN", "ATTACK"],
    digits=4,
    zero_division=0
))

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["BENIGN", "ATTACK"]
)
disp.plot(cmap="Blues", values_format=",")
plt.title("Random Forest — stratified random test split")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives:  {tn:,}")
print(f"False positives: {fp:,}")
print(f"False negatives: {fn:,}")
print(f"True positives:  {tp:,}")

## 11. Error analysis by original attack label

This breakdown shows performance for each original attack family in the random test subset. It is a diagnostic analysis; the trained model is still a binary classifier.

In [ ]:
test_results = pd.DataFrame({
    "Label": label_test.to_numpy(),
    "y_true": y_test.to_numpy(),
    "y_pred": y_test_pred
})

attack_test_results = test_results[test_results["y_true"] == 1].copy()

attack_wise = attack_test_results.groupby("Label").agg(
    rows=("y_true", "size"),
    detected=("y_pred", "sum")
)
attack_wise["missed"] = attack_wise["rows"] - attack_wise["detected"]
attack_wise["recall"] = attack_wise["detected"] / attack_wise["rows"]

display(attack_wise.sort_values("recall"))

## 12. Random Forest feature importance

Impurity-based feature importance can distribute importance across correlated features. Treat this as a model diagnostic, not proof that a feature causes an attack.

In [ ]:
rf_classifier = model.named_steps["classifier"]
importance = pd.Series(
    rf_classifier.feature_importances_,
    index=feature_columns,
    name="importance"
).sort_values(ascending=False)

display(importance.head(20).to_frame())

plt.figure(figsize=(9, 7))
importance.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 Random Forest feature importances")
plt.xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()

## 13. Save the model for FastAPI

The artifact includes:
- the fitted preprocessing/model pipeline,
- the selected decision threshold,
- the exact feature names and order,
- the binary target mapping.

The API must construct the same input features and use the same feature order.

In [ ]:
os.makedirs("models", exist_ok=True)

model_artifact = {
    "model": model,
    "threshold": selected_threshold,
    "feature_columns": feature_columns,
    "target_mapping": {"BENIGN": 0, "ATTACK": 1},
    "model_type": "RandomForest",
    "split": {
        "method": "stratified_random",
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "random_state": RANDOM_STATE
    },
    "evaluation": test_metrics
}

model_path = os.path.join("models", "sentinel_rf.pkl")
joblib.dump(model_artifact, model_path)

print("Model saved successfully:", os.path.abspath(model_path))

## 14. Results and limitations

### Latest recorded run

The run reported during project development produced:

| Metric | Test result |
|---|---:|
| Threshold | 0.05 |
| Accuracy | 98.68% |
| Attack precision | 93.71% |
| Attack recall | 100.00% |
| Attack F1 | 96.75% |
| True negatives | 335,350 |
| False positives | 5,615 |
| False negatives | 3 |
| True positives | 83,644 |

These values are a record of the reported run. Rerunning the notebook may produce slightly different results if the downloaded data or software environment changes.

### What this result supports

- The Random Forest baseline performed strongly on this stratified random CIC-IDS2017 test subset.
- The selected threshold achieved high attack recall on this test subset.
- The fitted pipeline and its feature contract were saved for backend integration.

### What this result does not establish

- Production-grade detection on live network traffic.
- Generalization to a different network or future traffic.
- Reliable detection of entirely unseen attack families.
- Zero false alarms: the reported test produced 5,615 benign flows flagged as attacks.

A separate time-based or dataset-level evaluation is needed before making real-world generalization claims.

## 15. Project checkpoint

At this stage, Sentinel Cyber has:
1. A reproducible CIC-IDS2017 loading and EDA workflow.
2. Numeric feature preparation and missing-value handling.
3. A stratified 70/15/15 train-validation-test split.
4. A trained Random Forest binary classifier.
5. Validation-based threshold selection.
6. Test metrics, confusion matrix, and attack-family error analysis.
7. A saved model artifact at `models/sentinel_rf.pkl`.

**Next integration step:** Load this artifact in FastAPI, validate incoming feature names, align them to `feature_columns`, call `model.predict_proba`, and compare the attack probability with the saved `threshold`.